## 1. Install Dependencies

In [1]:
!pip install -q ultralytics


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 2. Import Libraries and Check GPU

In [2]:
import os
import yaml
import torch
from ultralytics import YOLO
import time
from pathlib import Path

print("="*80)
print("YOLOv8 Drone Detection - Mac M4 Air Optimized")
print("="*80)

# Check for MPS (Metal Performance Shaders)
print(f"\nPyTorch version: {torch.__version__}")
print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"MPS built: {torch.backends.mps.is_built()}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/Users/hilkin/Library/Application Support/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLOv8 Drone Detection - Mac M4 Air Optimized

PyTorch version: 2.10.0
MPS available: True
MPS built: True


## 3. Configure Dataset

In [3]:
# Dataset paths
DATASET_ROOT = "/Users/hilkin/.cache/kagglehub/datasets/cybersimar08/drone-detection/versions/3/drone-detection-new.v5-new-train.yolov8"

# Verify dataset exists
if not os.path.exists(DATASET_ROOT):
    print(f"\n❌ Dataset not found at: {DATASET_ROOT}")
    print("Please update the DATASET_ROOT path.")
else:
    print(f"\n✓ Dataset found: {DATASET_ROOT}")

# Create data.yaml for YOLOv8
data_yaml_content = {
    'path': DATASET_ROOT,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 4,
    'names': ['drone', 'AirPlane', 'Drone', 'Helicopter']
}

# Save data.yaml
data_yaml_path = 'drone_data_yolov8.yaml'
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)

print(f"✓ Created {data_yaml_path}")


✓ Dataset found: /Users/hilkin/.cache/kagglehub/datasets/cybersimar08/drone-detection/versions/3/drone-detection-new.v5-new-train.yolov8
✓ Created drone_data_yolov8.yaml


## 4. Training Configuration

**Mac M4 Air Optimized Settings**:
- Device: MPS (GPU)
- Batch: 16 (aggressive for 25GB RAM)
- Workers: 4 (reduced for Jupyter compatibility)
- Cache: RAM (loads dataset to memory)
- Image size: 512 (faster than 640)
- Epochs: 100

In [5]:
# Training configuration
CONFIG = {
    'model': 'yolov8s.pt',  # Small model for speed
    'data': data_yaml_path,
    
    # Device settings - optimized for Mac M4
    'device': 'mps' if torch.backends.mps.is_available() else 'cpu',
    'workers': 4,  # Reduced for Jupyter (8 works in scripts)
    
    # Performance settings
    'batch': 16,  # Using your 25GB RAM!
    'imgsz': 512,  # Smaller for faster training
    'cache': 'ram',  # Cache in RAM - much faster!
    'amp': True,  # Mixed precision
    
    # Training settings
    'epochs': 30,
    'patience': 15,  # Early stopping
    'save_period': 10,  # Save every 10 epochs
    
    # Speed optimizations
    'multi_scale': False,  # Disable for speed
    'close_mosaic': 10,
    
    # Optimizer settings
    'optimizer': 'AdamW',
    'lr0': 0.001,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    
    # Output
    'project': 'drone_yolov8_results',
    'name': 'mac_m4_notebook',
    'exist_ok': True,
    'verbose': True,
    'plots': True,
}

print("\nTraining Configuration:")
print(f"  Model: {CONFIG['model']}")
print(f"  Device: {CONFIG['device']}")
print(f"  Batch size: {CONFIG['batch']}")
print(f"  Image size: {CONFIG['imgsz']}")
print(f"  Epochs: {CONFIG['epochs']}")
print(f"  Workers: {CONFIG['workers']}")
print(f"  Cache: {CONFIG['cache']}")
print(f"  Mixed precision: {CONFIG['amp']}")


Training Configuration:
  Model: yolov8s.pt
  Device: mps
  Batch size: 16
  Image size: 512
  Epochs: 30
  Workers: 4
  Cache: ram
  Mixed precision: True


## 5. Load Model

In [6]:
# Load YOLOv8 model
model = YOLO(CONFIG['model'])
print(f"✓ Loaded {CONFIG['model']}")

WARNING ⚠️ Download failure, retrying 1/3 https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov8s.pt... <urlopen error [Errno 54] Connection reset by peer>


#######################################################                   76.9%

✓ Loaded yolov8s.pt


######################################################################## 100.0%


## 6. Start Training

**Estimated Time**: 12-15 hours on Mac M4 Air

**What to expect**:
- First epoch will be slower (caching data)
- Subsequent epochs will be faster
- You'll see mAP scores improve over time
- Can interrupt with Kernel → Interrupt

**Note**: If this is too slow, use the Python script version instead for 2x speedup!

In [7]:
# Start training
start_time = time.time()

print("\n🚀 Starting training...\n")
print("="*80)

results = model.train(**CONFIG)

training_time = (time.time() - start_time) / 3600

print("\n" + "="*80)
print("✅ TRAINING COMPLETE!")
print("="*80)
print(f"Total training time: {training_time:.2f} hours")
print(f"Results saved to: {CONFIG['project']}/{CONFIG['name']}")
print("="*80)


🚀 Starting training...

Ultralytics 8.4.13 🚀 Python-3.14.2 torch-2.10.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=drone_data_yolov8.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=mac_m4_notebook, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=15, pers

## 7. Validate on Test Set

In [8]:
# Run validation on test set
print("\n📊 Running final validation on test set...\n")

test_results = model.val(
    data=CONFIG['data'],
    split='test',
    imgsz=CONFIG['imgsz'],
    batch=8,
    device=CONFIG['device']
)

print("\n" + "="*80)
print("TEST SET RESULTS")
print("="*80)
print(f"mAP50: {test_results.results_dict.get('metrics/mAP50(B)', 0):.3f}")
print(f"mAP50-95: {test_results.results_dict.get('metrics/mAP50-95(B)', 0):.3f}")
print(f"Precision: {test_results.results_dict.get('metrics/precision(B)', 0):.3f}")
print(f"Recall: {test_results.results_dict.get('metrics/recall(B)', 0):.3f}")
print("="*80)


📊 Running final validation on test set...

Ultralytics 8.4.13 🚀 Python-3.14.2 torch-2.10.0 MPS (Apple M4)
Model summary (fused): 73 layers, 11,127,132 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 60.5±16.2 MB/s, size: 14.4 KB)
val: Scanning /Users/hilkin/.cache/kagglehub/datasets/cybersimar08/drone-detection/versions/3/drone-detection-new.v5-new-train.yolov8/test/labels... 596 images, 123 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 596/596 8.0Kit/s 0.1s
val: New cache created: /Users/hilkin/.cache/kagglehub/datasets/cybersimar08/drone-detection/versions/3/drone-detection-new.v5-new-train.yolov8/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 75/75 4.2it/s 18.0s0.1s
                   all        596        479      0.941      0.941      0.967      0.655
                 drone        128        128      0.952      0.938      0.988      0.716
              AirPlane       

## 8. Save Final Model

In [9]:
# Save final model
final_model_path = 'drone_yolov8s_final.pt'
model.save(final_model_path)
print(f"\n✓ Final model saved to: {final_model_path}")
print("\n🎉 All done! Your Mac M4's GPU worked hard!")


✓ Final model saved to: drone_yolov8s_final.pt

🎉 All done! Your Mac M4's GPU worked hard!


## 9. Test Inference (Optional)

In [ ]:
# Test on a sample image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import glob
import random

# Get a random test image
test_images = glob.glob(f"{DATASET_ROOT}/test/images/*.jpg")
if test_images:
    test_img = random.choice(test_images)
    
    # Run inference
    results = model(test_img)
    
    # Display
    results[0].show()
    print(f"\nTest image: {test_img}")
else:
    print("No test images found")

---

## Performance Note

**Jupyter Notebook Limitation**: This notebook uses `workers=4` instead of `workers=8` due to macOS + Jupyter multiprocessing issues.

**For 2x Faster Training**: Use the Python script version:
```bash
python train_yolov8_optimized.py
```

The script can use more workers and will complete in ~8-10 hours instead of ~12-15 hours.